[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/STAT3009_ML_II.ipynb)

# Notebook 04: Machine Learning II — Generalization and model selection

**STAT3009 · Recommender Systems**  
**Suggested time:** 105–120 minutes of guided coding and discussion

ML I showed how to fit a model and calculate RMSE. This notebook asks a different question: **how do we choose a model setting without using the final test answers?**

```text
training data → holdout / cross-validation → choose hyperparameters
              → refit on all training data → evaluate once on test
```

We use the same two examples as the ML II slides: Ridge regression on California housing, then weighted three-mean prediction on the course's binary rating split. By the end, you should be able to:

1. distinguish fitting, validation, and final test roles;
2. identify Ridge's model, hyperparameter, fitting criterion, and learned parameters;
3. select `alpha` with `GridSearchCV` and read its results;
4. diagnose training and validation error across Ridge settings;
5. choose a rating-data split for the prediction situation, then select three-mean weights without leakage.

## 1. Setup

We reuse the libraries and RMSE metric from ML I. The new tools are `Ridge`, `train_test_split`, and `GridSearchCV`.

In [ ]:
import random

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split

pd.set_option('display.max_rows', 12)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 2. Ridge: the four questions

ML I used linear regression for California housing. Ridge keeps the same prediction rule but changes how the coefficients are fitted.

| Question | Ridge answer |
|---|---|
| **Model** | $\widehat y=b+x^{\mathsf T}w$ |
| **Hyperparameter** | $\alpha\geq0$, chosen *before* fitting |
| **Fitting criterion** | $\displaystyle\min_{b,w}\sum_{j=1}^{n}(y_j-b-x_j^{\mathsf T}w)^2+\alpha\lVert w\rVert_2^2$ |
| **Learned parameters** | $\widehat b$ and $\widehat w$, stored as `intercept_` and `coef_` after `fit` |

The penalty acts on $w$, not the intercept $b$. Training loss helps fit $b,w$ for one `alpha`; **validation RMSE** helps us choose among different `alpha` values. [Ridge documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).

### Reload the housing split from ML I

This notebook runs independently, so we reload the same fixed course split. The test arrays stay out of all model-selection cells.

In [ ]:
HOUSING_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/housing'
)

housing_train = pd.read_csv(f'{HOUSING_BASE_URL}/train.csv')
housing_test = pd.read_csv(f'{HOUSING_BASE_URL}/test.csv')

feature_columns = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms',
    'Population', 'AveOccup', 'Latitude', 'Longitude',
]

X_train_housing = housing_train[feature_columns].to_numpy(dtype=float)
y_train_housing = housing_train['MedHouseVal'].to_numpy(dtype=float)
X_test_housing = housing_test[feature_columns].to_numpy(dtype=float)
y_test_housing = housing_test['MedHouseVal'].to_numpy(dtype=float)

print('training rows:', len(y_train_housing))
print('test rows:', len(y_test_housing))
print('features:', X_train_housing.shape[1])

### One fit, one prediction

Changing `alpha` creates a different fitted model. Compare the constructor setting with the attributes learned from training rows.

In [ ]:
ridge_demo = Ridge(alpha=1.0)
print('alpha before fit:', ridge_demo.get_params()['alpha'])

ridge_demo.fit(X_train_housing, y_train_housing)
print('intercept_ after fit:', ridge_demo.intercept_)
print('coef_ after fit:', ridge_demo.coef_)
print('first three predictions:', ridge_demo.predict(X_train_housing[:3]))

### Checkpoint

If we change `alpha` from `1` to `100`, which value is chosen by us, and which values must be learned again? Why is training RMSE alone insufficient to choose `alpha`?

## 3. Holdout validation

Use only the **training** rows from the course split. `train_test_split` creates a smaller fitting set and a temporary validation set. Each candidate `alpha` is fitted from the same fitting rows and evaluated on the same validation rows.

In [ ]:
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train_housing,
    y_train_housing,
    test_size=0.25,
    random_state=42,
)

print('fit rows:', len(y_fit))
print('validation rows:', len(y_valid))
print('final test rows, still untouched:', len(y_test_housing))

In [ ]:
alpha_candidates = [0.1, 1.0, 100.0]
holdout_rows = []

for alpha in alpha_candidates:
    model = Ridge(alpha=alpha)
    model.fit(X_fit, y_fit)

    train_rmse = root_mean_squared_error(
        y_fit, model.predict(X_fit)
    )
    valid_rmse = root_mean_squared_error(
        y_valid, model.predict(X_valid)
    )
    holdout_rows.append({
        'alpha': alpha,
        'fit RMSE': train_rmse,
        'validation RMSE': valid_rmse,
    })

holdout_results = pd.DataFrame(holdout_rows)
display(holdout_results.sort_values('validation RMSE'))

### Discuss

1. Which `alpha` has the lowest **validation** RMSE? Does it also have the lowest fitting RMSE?
2. Would the ranking necessarily stay the same if we changed `random_state`? A single split can be noisy.
3. Did any test answer enter the table above?

## 4. Five-fold CV with `GridSearchCV`

Cross-validation rotates which training block acts as validation. For each `alpha`, `GridSearchCV` fits five times and averages five validation scores. `refit=True` then fits the winning `alpha` once more on **all** housing training rows. [GridSearchCV documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Scikit-learn **maximizes** scores, so its RMSE scorer is named `neg_root_mean_squared_error`. Change the sign back when reading `best_score_` or `mean_test_score`.

### Your turn: run a Ridge search

Fill in the candidate list, metric, and CV count. Use the **course training split**, not `X_fit` or the final test set. Then compare your cell with the solution below.

In [ ]:
# ridge_search = GridSearchCV(
#     estimator=Ridge(),
#     param_grid={'alpha': [...]},       # 0.1, 1.0, 100.0
#     scoring='...',                     # negative RMSE scorer
#     cv=...,                            # five folds
#     refit=True,
# )
# ridge_search.fit(X_train_housing, y_train_housing)

### Solution

In [ ]:
ridge_search = GridSearchCV(
    estimator=Ridge(),
    param_grid={'alpha': [0.1, 1.0, 100.0]},
    scoring='neg_root_mean_squared_error',
    cv=5,
    refit=True,
)
ridge_search.fit(X_train_housing, y_train_housing)

ridge_cv_results = pd.DataFrame({
    'alpha': ridge_search.cv_results_['param_alpha'],
    'CV RMSE': -ridge_search.cv_results_['mean_test_score'],
})
display(ridge_cv_results.sort_values('CV RMSE'))

### Read the selected estimator

`best_params_` gives the chosen **hyperparameter**. `best_score_` summarizes validation performance. `best_estimator_` contains coefficients learned by the **final refit**; they are not one fold's coefficients.

In [ ]:
print('chosen alpha:', ridge_search.best_params_['alpha'])
print(f'five-fold CV RMSE: {-ridge_search.best_score_:.4f}')

final_ridge = ridge_search.best_estimator_
print('final model alpha:', final_ridge.alpha)
print('final model intercept_:', final_ridge.intercept_)
print('final model coef_ shape:', final_ridge.coef_.shape)

### Diagnostic: fitting error versus CV error

The slide's curve uses **200 rows sampled from the housing training data** so that the fitting–validation gap is visible. This is a diagnostic example, **not** the full-data search that chose `final_ridge` above. The final test rows remain untouched.

We ask `GridSearchCV` for its optional training-fold scores with `return_train_score=True`. The same negative-RMSE sign convention applies to both training and validation columns. `refit=False` is appropriate here because we only want to inspect the curve; the earlier full-data search already refitted its winner.

In [ ]:
import matplotlib.pyplot as plt

demo_rows = random.Random(42).sample(range(len(y_train_housing)), 200)
X_demo = X_train_housing[demo_rows]
y_demo = y_train_housing[demo_rows]

curve_search = GridSearchCV(
    estimator=Ridge(),
    param_grid={'alpha': [0.1, 1, 10, 100, 1000, 10000, 100000]},
    scoring='neg_root_mean_squared_error',
    cv=5,
    refit=False,
    return_train_score=True,
)
curve_search.fit(X_demo, y_demo)

curve_results = pd.DataFrame({
    'alpha': np.asarray(curve_search.cv_results_['param_alpha'], dtype=float),
    'fit-fold RMSE': -curve_search.cv_results_['mean_train_score'],
    'validation-fold RMSE': -curve_search.cv_results_['mean_test_score'],
})
display(curve_results)

ax = curve_results.plot(
    x='alpha', y=['fit-fold RMSE', 'validation-fold RMSE'],
    marker='o', logx=True, figsize=(8, 4),
)
ax.set_ylabel('RMSE (lower is better)')
ax.grid(axis='y', alpha=0.3)
plt.show()

### Discuss the curve

1. Which `alpha` gives the smallest **fitting** RMSE? Which gives the smallest **validation** RMSE in this diagnostic sample?
2. What happens to both errors when `alpha` becomes very large?
3. Why should we **not** replace the earlier full-data `final_ridge` with this 200-row diagnostic model?

A CV curve need not always have a U shape. Read the measured values before drawing a conclusion.

### Final housing evaluation

The choice is now fixed. Predict the separate test rows and report RMSE once. The housing target is measured in units of $100{,}000$.

In [ ]:
housing_test_pred = final_ridge.predict(X_test_housing)
housing_test_rmse = root_mean_squared_error(
    y_test_housing, housing_test_pred
)
print(f'final housing test RMSE: {housing_test_rmse:.4f}')

## 5. Transfer the workflow to three-mean recommendation

We now choose the weights of the three-mean predictor from ML I:

$$\widehat r_{ui}=w_u\bar r_u+w_i\bar r_i+w_0\mu,\qquad w_u+w_i+w_0=1.$$

The **three means** are learned from ratings in each fitting fold. The **weights** are chosen by validation. If a validation pair has an unseen user or item, that missing mean uses the fitting fold's global mean.

We use `dataset/b-rs/train.csv`, the same **0/1 binary rating** split used for the slide's weight-comparison table. The earlier Netflix example used 1–5 ratings; the prediction rule is the same, but the RMSE scale is different. These IDs are already integer-coded, so no new encoding step is needed.

In [ ]:
RS_BASE_URL = (
    'https://raw.githubusercontent.com/'
    'statmlben/CUHK-STAT3009/main/dataset/b-rs'
)

rs_train = pd.read_csv(f'{RS_BASE_URL}/train.csv')
X_train_rs = rs_train[['user_id', 'movie_id']].to_numpy(dtype=int)
y_train_rs = rs_train['rating'].to_numpy(dtype=float)

n_users = int(X_train_rs[:, 0].max()) + 1
n_items = int(X_train_rs[:, 1].max()) + 1

print('training ratings:', len(y_train_rs))
print('X shape:', X_train_rs.shape)
print('observed rating values:', sorted(set(y_train_rs.tolist())))
display(rs_train[['user_id', 'movie_id', 'rating']].head())

### Give each observed pair a fold

We shuffle row positions once, then assign every fifth shuffled position to the same fold. Thus all four weight candidates will use **identical** validation folds. The seed `42` reproduces the folds in the slides.

In [ ]:
shuffled_rows = list(range(len(y_train_rs)))
random.Random(42).shuffle(shuffled_rows)

fold_id = np.empty(len(y_train_rs), dtype=int)
for fold in range(5):
    fold_id[shuffled_rows[fold::5]] = fold

print('ratings per fold:', [int((fold_id == j).sum()) for j in range(5)])

### What does a pair-level fold actually test?

In the first fold, an ID is **known** only if it appears in the other four fitting folds. A random pair split mostly creates new pairs for familiar users and items, but rare IDs may appear only in validation. Count the four cases before fitting the three means.

In [ ]:
first_fit_rows = fold_id != 0
first_valid_rows = fold_id == 0
users_in_fit = set(X_train_rs[first_fit_rows, 0])
items_in_fit = set(X_train_rs[first_fit_rows, 1])
X_first_valid = X_train_rs[first_valid_rows]

known_user = np.array([u in users_in_fit for u in X_first_valid[:, 0]])
known_item = np.array([i in items_in_fit for i in X_first_valid[:, 1]])

print('known user + known item:', int((known_user & known_item).sum()))
print('new user + known item:', int((~known_user & known_item).sum()))
print('known user + new item:', int((known_user & ~known_item).sum()))
print('new user + new item:', int((~known_user & ~known_item).sum()))

### Choose the validation question

1. What fraction of these validation rows has both IDs known? Which cases will dominate one overall pair-level RMSE?
2. If the application must recommend to **entirely new users**, should we hold out random pairs or all ratings from selected users?
3. Which mean is unavailable for a new user? For a new item?

We keep the current pair-level folds for today's weight comparison. A whole-user or whole-item holdout would answer a **different** deployment question; it is not another weight candidate.

### Prepare fold-specific means

This is supplied setup code, using the same `set`, Boolean masks, `np.full`, and `.mean()` pattern as the previous notebook. Read its **order**: split rows → compute means from fitting rows → look up means for validation pairs. Do not compute means from the whole training file before making folds.

For each fold, we save the three numbers needed to predict every validation pair. The next exercise only changes their weights.

In [ ]:
fold_terms = []

for fold in range(5):
    fit_rows = fold_id != fold
    valid_rows = fold_id == fold
    X_fold_fit = X_train_rs[fit_rows]
    y_fold_fit = y_train_rs[fit_rows]
    X_fold_valid = X_train_rs[valid_rows]
    y_fold_valid = y_train_rs[valid_rows]

    global_mean = y_fold_fit.mean()
    user_mean = np.full(n_users, global_mean)
    item_mean = np.full(n_items, global_mean)

    for user in set(X_fold_fit[:, 0]):
        user_mean[user] = y_fold_fit[X_fold_fit[:, 0] == user].mean()
    for item in set(X_fold_fit[:, 1]):
        item_mean[item] = y_fold_fit[X_fold_fit[:, 1] == item].mean()

    fold_terms.append({
        'y_valid': y_fold_valid,
        'user': user_mean[X_fold_valid[:, 0]],
        'item': item_mean[X_fold_valid[:, 1]],
        'global': global_mean,
    })

print('folds prepared:', len(fold_terms))
print('validation rows in first fold:', len(fold_terms[0]['y_valid']))

### One prediction before the search

Inspect a single held-out pair. All three mean values were learned without its rating. The `y_valid` value is used **only after** prediction, when calculating RMSE.

In [ ]:
first_fold = fold_terms[0]
first_equal_prediction = (
    first_fold['user'][0]
    + first_fold['item'][0]
    + first_fold['global']
) / 3

print('user mean:', first_fold['user'][0])
print('item mean:', first_fold['item'][0])
print('global mean:', first_fold['global'])
print('equal-weight prediction:', first_equal_prediction)
print('held-out rating:', first_fold['y_valid'][0])

## 6. In-class practice: select three-mean weights

Try the four candidates from the slides. For **each candidate and each fold**, combine the stored means, calculate RMSE, and average the five fold RMSEs. The result should be close to the slide table (four decimal places).

| Candidate | `w_user` | `w_item` | `w_global` |
|---|---:|---:|---:|
| Equal | 1/3 | 1/3 | 1/3 |
| User-heavy | 0.5 | 0.3 | 0.2 |
| Item-heavy | 0.3 | 0.5 | 0.2 |
| Global-heavy | 0.2 | 0.2 | 0.6 |

**Coding goal:** write the weighted prediction expression and the fold RMSE calculation. The outer loop structure is supplied.

In [ ]:
weight_candidates = {
    'Equal': (1/3, 1/3, 1/3),
    'User-heavy': (0.5, 0.3, 0.2),
    'Item-heavy': (0.3, 0.5, 0.2),
    'Global-heavy': (0.2, 0.2, 0.6),
}

# cv_scores = {}
# for name, (w_user, w_item, w_global) in weight_candidates.items():
#     fold_scores = []
#     for terms in fold_terms:
#         predictions = (                    # combine the three terms
#             ...
#         )
#         fold_rmse = ...                    # compare y_valid and predictions
#         fold_scores.append(fold_rmse)
#     cv_scores[name] = fold_scores
#
# score_table = pd.DataFrame(cv_scores).T
# score_table.columns = [f'Fold {j}' for j in range(1, 6)]
# score_table['CV mean'] = score_table.mean(axis=1)
# display(score_table.sort_values('CV mean'))

### Solution

In [ ]:
cv_scores = {}

for name, (w_user, w_item, w_global) in weight_candidates.items():
    fold_scores = []
    for terms in fold_terms:
        predictions = (
            w_user * terms['user']
            + w_item * terms['item']
            + w_global * terms['global']
        )
        fold_rmse = root_mean_squared_error(
            terms['y_valid'], predictions
        )
        fold_scores.append(fold_rmse)
    cv_scores[name] = fold_scores

score_table = pd.DataFrame(cv_scores).T
score_table.columns = [f'Fold {j}' for j in range(1, 6)]
score_table['CV mean'] = score_table.mean(axis=1)
display(score_table.sort_values('CV mean'))

### Discuss

1. Which candidate has the lowest average validation RMSE? Is every individual fold won by the same candidate?
2. Which quantities changed from fold to fold: the three means, the weights, or both?
3. Why would using a mean computed from **all** 51,161 ratings make these CV scores too optimistic?

## 7. Refit the selected three-mean model

Selection is complete. Recompute the user, item, and global means using **all training ratings**, then use the selected weights. This is the same fit–validate–refit sequence that `GridSearchCV` carried out automatically for Ridge.

In [ ]:
best_name = score_table['CV mean'].idxmin()
best_w_user, best_w_item, best_w_global = weight_candidates[best_name]

final_global_mean = y_train_rs.mean()
final_user_mean = np.full(n_users, final_global_mean)
final_item_mean = np.full(n_items, final_global_mean)

for user in set(X_train_rs[:, 0]):
    final_user_mean[user] = y_train_rs[X_train_rs[:, 0] == user].mean()
for item in set(X_train_rs[:, 1]):
    final_item_mean[item] = y_train_rs[X_train_rs[:, 1] == item].mean()

print('selected candidate:', best_name)
print('selected weights:', (best_w_user, best_w_item, best_w_global))
print(f'five-fold CV RMSE: {score_table.loc[best_name, "CV mean"]:.4f}')

### Final classroom evaluation

The published `test.csv` includes answers for teaching. We load it **only after** choosing the weights and use it once to demonstrate final evaluation. A real competition test set would not reveal these ratings.

In [ ]:
rs_test = pd.read_csv(f'{RS_BASE_URL}/test.csv')
X_test_rs = rs_test[['user_id', 'movie_id']].to_numpy(dtype=int)
y_test_rs_demo = rs_test['rating'].to_numpy(dtype=float)

# Start with the global fallback. Overwrite entries for IDs seen in training.
user_terms_test = np.full(len(X_test_rs), final_global_mean)
item_terms_test = np.full(len(X_test_rs), final_global_mean)

known_user = (X_test_rs[:, 0] >= 0) & (X_test_rs[:, 0] < n_users)
known_item = (X_test_rs[:, 1] >= 0) & (X_test_rs[:, 1] < n_items)
user_terms_test[known_user] = final_user_mean[X_test_rs[known_user, 0]]
item_terms_test[known_item] = final_item_mean[X_test_rs[known_item, 1]]

rs_test_pred = (
    best_w_user * user_terms_test
    + best_w_item * item_terms_test
    + best_w_global * final_global_mean
)
rs_test_rmse = root_mean_squared_error(y_test_rs_demo, rs_test_pred)

print('test predictions:', rs_test_pred.shape)
print(f'final classroom test RMSE: {rs_test_rmse:.4f}')

## Exit ticket

1. In Ridge, what is chosen before fitting, and what is learned during fitting?
2. What does `-ridge_search.best_score_` mean? Why do we change the sign?
3. In the recommender example, why must the user and item means be recomputed **inside each fold**?
4. After choosing three-mean weights, why do we fit the means again using all training ratings?
5. What would go wrong if we chose another candidate because it happened to score better on the final classroom test?

```text
Define task and metric → choose with validation → refit → test once
```